# Exercícios Aula 02: Inteligência Visual

Este notebook contém **5 exercícios práticos** para reforçar os principais conceitos apresentados nos notebooks da Aula 02 (`02_01` e `02_02`):

1. Pipeline de classificação com o módulo DNN do OpenCV (blob + inferência)
2. Diferenças de pré-processamento entre modelos (MobileNetV2 x EfficientNet)
3. Detecção de objetos com YOLO
4. Segmentação de objetos com YOLO-seg
5. Segmentação guiada por ponto com SAM

Complete os blocos marcados com `# TODO` em cada exercício. Use os notebooks `02_01` e `02_02` como referência sempre que precisar relembrar a sintaxe de alguma função.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93
    !pip install opencv-contrib-python==5.0.0.93
    !pip install ultralytics
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos, vamos utilizar `pathlib` e `urllib` (download de modelos) e `ultralytics` (família YOLO e SAM).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from ultralytics import YOLO, SAM

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline
from IPython.display import Image

## Preparação: Modelos de Classificação

Antes dos exercícios 1 e 2, vamos reaproveitar a função de download e carregar os dois modelos de classificação vistos no notebook `02_01` (MobileNetV2 e EfficientNet), além da lista de classes do ImageNet. Essa célula é apenas infraestrutura — não faz parte dos exercícios.

In [ ]:
def baixar_arquivo_modelo(url: str, nome_arquivo: str) -> Path:
    # Pasta para armazenar os modelos
    MODELOS_DIR = Path('modelos')
    MODELOS_DIR.mkdir(exist_ok=True)

    path_modelo = MODELOS_DIR / nome_arquivo

    if not path_modelo.exists():
        print("Baixando arquivo...")
        urlretrieve(url, path_modelo)
        print("Download concluído!")
    else:
        print("Arquivo já existe.")

    return path_modelo

# Carregando MobileNetV2
url_mobilenetv2 = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/mobilenet/model/mobilenetv2-12.onnx"
)
net_mobilenetv2 = cv2.dnn.readNet(str(baixar_arquivo_modelo(url_mobilenetv2, 'mobilenetv2-12.onnx')))

# Carregando EfficientNet
url_efficientnet = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/efficientnet-lite4/model/efficientnet-lite4-11.onnx"
)
net_efficientnet = cv2.dnn.readNet(str(baixar_arquivo_modelo(url_efficientnet, 'efficientnet-lite4-11.onnx')))

# Carregando lista de classes do ImageNet
url_classes = (
    "https://github.com/onnx/models/raw/refs/heads/"
    "main/validated/vision/classification/synset.txt"
)
classes_arquivo = baixar_arquivo_modelo(url_classes, 'synset.txt')
with open(classes_arquivo, 'r') as f:
    classes = [linha.strip().split(maxsplit=1)[-1] for linha in f]

print("Modelos e classes carregados com sucesso!")

## Exercício 1: Blob e Inferência com MobileNetV2

**Conceito reforçado:** por que uma imagem precisa ser convertida em **blob** antes de entrar na rede (redimensionamento, normalização, troca de canais BGR → RGB) e como interpretar a saída de um classificador (vetor de scores → `argmax` → classe prevista).

1. Leia a imagem `imagens/02/iron-maiden.webp` com OpenCV.
2. Converta a imagem para blob usando `cv2.dnn.blobFromImage()`, com `scalefactor=1/255.`, `size=(224, 224)`, `mean=(0.485, 0.456, 0.406)` e `swapRB=True` (os mesmos parâmetros usados para o MobileNetV2 em `02_01`). Não esqueça de ajustar o desvio padrão do ImageNet dividindo os canais do blob por `(0.229, 0.224, 0.225)`.
3. Rode a inferência com `net_mobilenetv2.setInput()` seguido de `.forward()`.
4. Descubra o índice de maior score (`np.argmax`) e imprima a classe (índice), a descrição (usando a lista `classes`) e a confiança.

In [ ]:
# 1. Leia a imagem 'imagens/02/iron-maiden.webp'


In [ ]:
# 2. Converta a imagem para blob (lembre-se de ajustar o desvio padrão ImageNet)


In [ ]:
# 3. Rode a inferência com net_mobilenetv2


In [ ]:
# 4. Descubra a classe de maior score e imprima classe, descrição e confiança


## Exercício 2: Comparando MobileNetV2 e EfficientNet

**Conceito reforçado:** cada modelo pode exigir um pré-processamento (blob) diferente — não existe uma única forma "certa" de montar o blob, ela depende de como o modelo foi treinado.

1. Usando a mesma imagem do Exercício 1, monte o blob no formato esperado pelo **EfficientNet**: converta BGR → RGB, redimensione para `(224, 224)`, converta para `float32`, escale os valores para o intervalo `[0, 1]` e adicione a dimensão de batch (formato NHWC — sem trocar para NCHW como fizemos com o MobileNetV2).
2. Rode a inferência com `net_efficientnet`.
3. Compare a classe prevista pelo EfficientNet com a classe prevista pelo MobileNetV2 no Exercício 1. Elas concordam? Escreva sua conclusão em um comentário.

In [ ]:
# 1. Monte o blob no formato esperado pelo EfficientNet


In [ ]:
# 2. Rode a inferência com net_efficientnet


In [ ]:
# 3. Compare com o resultado do MobileNetV2 (Exercício 1) e escreva sua conclusão:


## Exercício 3: Detecção de Objetos com YOLO

**Conceito reforçado:** ao contrário do módulo DNN do OpenCV, a biblioteca `ultralytics` cuida da conversão para blob automaticamente. O resultado de uma detecção retorna uma lista de caixas (`boxes`), cada uma com classe, confiança e coordenadas.

1. Carregue o modelo `YOLO('modelos/yolo26n.pt')`.
2. Leia a imagem `imagens/02/futebol.webp` e converta para RGB.
3. Rode a inferência (`detect_model(img_rgb)`).
4. Ao invés de usar `results[0].plot()`, percorra manualmente `results[0].boxes` e conte quantos objetos foram detectados de cada classe (dica: use um dicionário para acumular a contagem, usando `results[0].names[classe]` como chave).

In [ ]:
# 1. Carregue o modelo YOLO de detecção


In [ ]:
# 2. Leia a imagem 'imagens/02/futebol.webp' e converta para RGB


In [ ]:
# 3. Rode a inferência


In [ ]:
# 4. Percorra results[0].boxes e conte quantos objetos existem de cada classe


## Preparação: Função de Apresentação de Segmentações

Antes dos exercícios 4 e 5, vamos reaproveitar a função `apresentar_segmentacoes()` criada no notebook `02_02`, que sobrepõe as máscaras de segmentação na imagem original.

In [ ]:
def apresentar_segmentacoes(img: np.ndarray, result) -> None:
    CORES = [
        (255,   0,   0),   # vermelho
        (  0, 255,   0),   # verde
        (  0,   0, 255),   # azul
        (255, 128,   0),   # laranja
        (255,   0, 255),   # magenta
        (  0, 255, 255),   # ciano
        (255, 255,   0),   # amarelo
        (128,   0, 255),   # roxo
        (255,   0, 128),   # rosa
        (  0, 128, 255),   # azul claro
        (128, 255,   0),   # verde limão
        (255,  64,  64),   # vermelho claro
    ]

    img_copy = img.copy()
    overlay = img.copy()

    for i, (box, mask) in enumerate(zip(result.boxes, result.masks.data)):
        # Escolhe cor
        cor = CORES[i % len(CORES)]

        # Prepara máscara
        mask = mask.cpu().numpy()
        mask = cv2.resize(
            mask,
            (img.shape[1], img.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

        mask = mask > 0.5
        overlay[mask] = cor

        # Apresenta classe e grau de confiança
        class_id = int(box.cls[0])
        label = result.names[class_id]
        conf = float(box.conf[0])

        x1, y1, x2, y2 = (
            box.xyxy[0]
            .cpu()
            .numpy()
            .astype(int)
        )

        cv2.rectangle(img_copy, (x1, y1), (x2, y2), cor, 2)
        cv2.putText(
            img_copy,
            f"{label} {conf:.2f}",
            (x1, y1-5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            cor,
            2
        )

    # Sobrepõe máscara na imagem
    resultado = cv2.addWeighted(overlay, 0.35, img_copy, 0.65, 0)

    # Apresenta
    plt.figure(figsize=(12, 8))
    plt.imshow(resultado)
    plt.axis('off')
    plt.show()

## Exercício 4: Isolando um Objeto com Segmentação (YOLO-seg)

**Conceito reforçado:** diferente da detecção (que retorna apenas uma caixa retangular), a segmentação retorna uma **máscara pixel a pixel** (`masks`) que indica exatamente quais pixels pertencem ao objeto.

1. Carregue o modelo `YOLO('modelos/yolo26n-seg.pt')`.
2. Rode a inferência na mesma imagem do Exercício 3 (`imagens/02/futebol.webp`).
3. Escolha um dos objetos detectados e pegue sua máscara em `result.masks.data[i]`.
4. Redimensione a máscara para o tamanho da imagem original (`cv2.resize()`) e use-a para "recortar" apenas aquele objeto da imagem (dica: aplique a máscara multiplicando-a pela imagem ou usando `cv2.bitwise_and()`), apresentando o resultado com o fundo preto.

In [ ]:
# 1. Carregue o modelo YOLO de segmentação


In [ ]:
# 2. Rode a inferência na imagem do Exercício 3


In [ ]:
# 3. Escolha um objeto e pegue sua máscara em result.masks.data[i]


In [ ]:
# 4. Redimensione a máscara e use-a para recortar o objeto (fundo preto)


## Exercício 5: Segmentação Guiada por Ponto com SAM

**Conceito reforçado:** diferente do YOLO (que reconhece apenas categorias fixas aprendidas no treinamento), o **SAM** não depende de categorias — ele segmenta qualquer objeto a partir de um ponto ou área indicados pelo usuário.

1. Carregue o modelo `SAM('modelos/mobile_sam.pt')`.
2. Leia a imagem `imagens/02/luke.jpg` e converta para RGB.
3. Escolha, por tentativa e erro, um ponto `[x, y]` sobre um objeto de seu interesse na imagem (por exemplo, o rosto da pessoa ou o cachorro).
4. Rode a inferência com `sam_model(img_rgb, points=[[x, y]])` e apresente a segmentação resultante usando a função `apresentar_segmentacoes()`.

In [ ]:
# 1. Carregue o modelo SAM


In [ ]:
# 2. Leia a imagem 'imagens/02/luke.jpg' e converta para RGB


In [ ]:
# 3. Escolha um ponto [x, y] sobre o objeto que deseja segmentar


In [ ]:
# 4. Rode a inferência com o ponto escolhido e apresente o resultado
